# 06 — Software Supply Chain Sentinel: Report

## The NPM Ecosystem Risk Atlas

---

> *"On December 9, 2021, a single CVE (Log4Shell, CVE-2021-44228) exposed approximately 35,000 Java packages that transitively depended on Log4j. Most of those package maintainers had no idea their software was at risk until attackers were already exploiting it. This project asks: could we have predicted the blast radius before the exploit?"*

---

This report presents the findings of a large-scale graph analysis of the NPM ecosystem, modeling **~50,000 packages** and their dependency and contributor relationships as a directed network. We compute four structural risk metrics and combine them into the **Sentinel Score** — a novel composite metric for identifying supply chain fragility.

**Key findings preview:**
- X packages in the top 500 by centrality have a Bus Factor of 1
- Y packages have known CVEs AND high Sentinel Scores ("critical" tier)
- Z packages are Orphan Risk — high download counts, ≤2 npm maintainers, not updated in >12 months
- The average NPM package in our top 10k has N direct dependents and M transitive dependents at 3 hops

*(These values are filled in after running the full analysis pipeline)*

## 0 — Setup

In [ ]:
!pip install -q pandas pyarrow matplotlib seaborn networkx pyvis

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import networkx as nx

BASE    = '/content/drive/MyDrive/BlastRadius'
OUT_DIR = f'{BASE}/data/processed'
FIG_DIR = f'{BASE}/figures'
os.makedirs(FIG_DIR, exist_ok=True)

# Consistent styling
plt.rcParams.update({
    'figure.dpi': 120,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'font.size': 11
})
RISK_COLORS = {'CRITICAL': '#d62728', 'HIGH': '#ff7f0e', 'MEDIUM': '#ffbb78', 'LOW': '#2ca02c', 'UNKNOWN': '#aec7e8'}

In [ ]:
# Load all analysis outputs
sentinel    = pd.read_parquet(f'{OUT_DIR}/sentinel_enriched.parquet')
pagerank    = pd.read_parquet(f'{OUT_DIR}/pagerank.parquet')
bus_factor  = pd.read_parquet(f'{OUT_DIR}/bus_factor.parquet')
orphan_risk = pd.read_parquet(f'{OUT_DIR}/orphan_risk.parquet')
communities = pd.read_parquet(f'{OUT_DIR}/communities.parquet')
blast_df    = pd.read_parquet(f'{OUT_DIR}/blast_radius.parquet')

sentinel['cve_ids'] = sentinel['cve_ids'].apply(
    lambda x: json.loads(x) if isinstance(x, str) else x
)

print(f'Packages in final table: {len(sentinel):,}')
print(f'Risk tier distribution:')
print(sentinel['risk_tier'].value_counts())

## 1 — Dataset & Methodology

In [ ]:
# Dataset summary statistics
total_pkgs      = len(sentinel)
pkgs_with_github = sentinel['github_slug'].notna().sum() if 'github_slug' in sentinel.columns else 'N/A'
pkgs_with_cve   = sentinel['has_cve'].sum()
pkgs_with_bf    = bus_factor['package_name'].nunique()
bf1_count       = (bus_factor['bus_factor'] == 1).sum()
orphan_flagged  = orphan_risk['is_orphan_risk'].sum()
critical_count  = sentinel['is_critical'].sum()

print('=' * 55)
print('DATASET SUMMARY')
print('=' * 55)
print(f'Total NPM packages analyzed:         {total_pkgs:>8,}')
print(f'Packages with GitHub repository:     {pkgs_with_github:>8,}')
print(f'Packages with Bus Factor data:       {pkgs_with_bf:>8,}')
print(f'Packages with ≥1 CVE:                {pkgs_with_cve:>8,}')
print(f'Packages flagged as Orphan Risk:     {orphan_flagged:>8,}')
print(f'CRITICAL packages (score+CVE):       {critical_count:>8,}')
print(f'Bus Factor = 1:                      {bf1_count:>8,}')

## 2 — Top 20: Sentinel Score Leaderboard

In [ ]:
top20 = sentinel.nlargest(20, 'sentinel_score').reset_index(drop=True)

fig, ax = plt.subplots(figsize=(12, 7))
colors = [RISK_COLORS.get(tier, '#aec7e8') for tier in top20['risk_tier']]

bars = ax.barh(
    top20['name'][::-1],
    top20['sentinel_score'][::-1],
    color=colors[::-1],
    edgecolor='white',
    linewidth=0.5
)

# Annotate bars with Bus Factor and CVE flag
for i, (_, row) in enumerate(top20[::-1].iterrows()):
    label = f'BF={int(row["bus_factor"])}'
    if row['has_cve']:
        label += ' ⚠CVE'
    ax.text(
        row['sentinel_score'] * 1.01, i,
        label, va='center', fontsize=8, color='#333333'
    )

# Legend
patches = [mpatches.Patch(color=v, label=k) for k, v in RISK_COLORS.items() if k != 'UNKNOWN']
ax.legend(handles=patches, title='Risk Tier', loc='lower right', fontsize=9)

ax.set_xlabel('Sentinel Score  (PageRank × 1/BusFactor × RecencyDecay)', fontsize=10)
ax.set_title('Top 20 NPM Packages by Sentinel Score', fontsize=13, fontweight='bold')
ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'{x:.3f}'))
plt.tight_layout()
plt.savefig(f'{FIG_DIR}/sentinel_top20.png', bbox_inches='tight')
plt.show()
print(f'Figure saved → {FIG_DIR}/sentinel_top20.png')

In [ ]:
# Table version
print('Sentinel Score Leaderboard — Top 20:')
display_cols = ['name', 'sentinel_score', 'pagerank', 'bus_factor', 'has_cve', 'risk_tier']
print(top20[display_cols].to_string(index=False, float_format='{:.4f}'.format))

## 3 — Top 20: Orphan Risk Leaderboard

In [ ]:
top20_orphan = orphan_risk.nlargest(20, 'orphan_score').reset_index(drop=True)

fig, ax = plt.subplots(figsize=(12, 7))

orphan_colors = [
    '#d62728' if m == 1 else ('#ff7f0e' if m == 2 else '#ffbb78')
    for m in top20_orphan['maintainer_count'][::-1]
]

ax.barh(
    top20_orphan['name'][::-1],
    top20_orphan['orphan_score'][::-1],
    color=orphan_colors,
    edgecolor='white',
    linewidth=0.5
)

for i, (_, row) in enumerate(top20_orphan[::-1].iterrows()):
    months = row.get('months_since_publish', 0)
    label = f"{row['maintainer_count']} maintainer(s), {months:.0f}mo stale"
    ax.text(
        row['orphan_score'] * 1.01, i,
        label, va='center', fontsize=8, color='#333333'
    )

patches = [
    mpatches.Patch(color='#d62728', label='1 maintainer'),
    mpatches.Patch(color='#ff7f0e', label='2 maintainers'),
    mpatches.Patch(color='#ffbb78', label='3+ maintainers (stale)'),
]
ax.legend(handles=patches, title='npm Maintainer Count', loc='lower right', fontsize=9)

ax.set_xlabel('Orphan Risk Score  (log(dependents) × staleness × 1/maintainers)', fontsize=10)
ax.set_title('Top 20 NPM Packages by Orphan Risk Score', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(f'{FIG_DIR}/orphan_risk_top20.png', bbox_inches='tight')
plt.show()
print(f'Figure saved → {FIG_DIR}/orphan_risk_top20.png')

## 4 — Community Cluster Visualization

In [ ]:
# Load graph edges for visualization
USE_SAMPLE = True
suffix = '_10k' if USE_SAMPLE else ''
DATA_DIR = f'{BASE}/data/sample' if USE_SAMPLE else f'{BASE}/data/processed'

edges_pd = pd.read_parquet(f'{DATA_DIR}/graph_edges{suffix}.parquet')
dep_edges = edges_pd[edges_pd['edge_type'] == 'DEPENDS_ON'][['src', 'dst']]

# Build graph for top 300 packages only (visualization readability)
top300_names = set(sentinel.nlargest(300, 'sentinel_score')['name'].str.lower())
top300_ids   = {'pkg:' + n for n in top300_names}

sub_edges = dep_edges[
    dep_edges['src'].isin(top300_ids) &
    dep_edges['dst'].isin(top300_ids)
]
G_vis = nx.DiGraph()
G_vis.add_edges_from(zip(sub_edges['src'], sub_edges['dst']))

# Remove isolated nodes
G_vis.remove_nodes_from(list(nx.isolates(G_vis)))
print(f'Visualization subgraph: {G_vis.number_of_nodes()} nodes, {G_vis.number_of_edges()} edges')

In [ ]:
# Assign community colors
comm_lookup = communities.set_index('id')['community_id'].to_dict()
sentinel_lookup = sentinel.set_index('name')['sentinel_score'].to_dict()

# Get top communities by size
community_ids_in_subgraph = [
    comm_lookup.get(n, -1) for n in G_vis.nodes
]
from collections import Counter
top_communities = [c for c, _ in Counter(community_ids_in_subgraph).most_common(8) if c != -1]
community_palette = plt.cm.Set2(np.linspace(0, 1, len(top_communities)))
community_color_map = {comm: community_palette[i] for i, comm in enumerate(top_communities)}

node_colors = [
    community_color_map.get(comm_lookup.get(n, -1), (0.7, 0.7, 0.7, 1.0))
    for n in G_vis.nodes
]

# Node sizes proportional to sentinel score
node_sizes = [
    200 + 3000 * sentinel_lookup.get(n.replace('pkg:', ''), 0.0)
    for n in G_vis.nodes
]

fig, ax = plt.subplots(figsize=(16, 12))
pos = nx.spring_layout(G_vis, seed=42, k=0.4, iterations=30)

nx.draw_networkx_nodes(
    G_vis, pos, ax=ax,
    node_color=node_colors,
    node_size=node_sizes,
    alpha=0.85
)
nx.draw_networkx_edges(
    G_vis, pos, ax=ax,
    edge_color='#cccccc',
    arrows=False,
    width=0.3,
    alpha=0.5
)

# Label only the top 30 nodes by sentinel score
top30_ids = {'pkg:' + n for n in sentinel.nlargest(30, 'sentinel_score')['name'].str.lower()}
top30_in_subgraph = {n: n.replace('pkg:', '') for n in G_vis.nodes if n in top30_ids}
nx.draw_networkx_labels(
    G_vis, pos, labels=top30_in_subgraph, ax=ax,
    font_size=7, font_weight='bold'
)

ax.set_title(
    'NPM Ecosystem Community Structure\n(node size = Sentinel Score, color = community cluster, top 300 packages by risk)',
    fontsize=12, fontweight='bold'
)
ax.axis('off')
plt.tight_layout()
plt.savefig(f'{FIG_DIR}/community_clusters.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'Figure saved → {FIG_DIR}/community_clusters.png')

## 5 — Case Study: Deep Dive on the Highest-Risk Package

In [ ]:
case_study = sentinel.iloc[0]

print('=' * 60)
print(f'CASE STUDY: {case_study["name"].upper()}')
print('=' * 60)
print(f'Sentinel Score:      {case_study["sentinel_score"]:.4f}  (#{1} overall)')
print(f'PageRank:            {case_study["pagerank"]:.4f}')
print(f'Bus Factor:          {int(case_study["bus_factor"])} active maintainer(s) in last 12 months')
print(f'Recency Decay:       {case_study["recency_decay"]:.3f}  (1.0 = active today, 0.0 = abandoned)')
print(f'Risk Tier:           {case_study["risk_tier"]}')
print(f'Has CVE:             {case_study["has_cve"]}')
if case_study['has_cve']:
    cve_list = case_study['cve_ids'] if isinstance(case_study['cve_ids'], list) else json.loads(case_study['cve_ids'])
    print(f'CVE Count:           {case_study["cve_count"]}')
    print(f'Worst Severity:      {case_study["max_severity"]}')
    print(f'CVE IDs:             {", ".join(cve_list[:5])}{" ..." if len(cve_list) > 5 else ""}')

# Blast radius from notebook 04
br = blast_df[blast_df['source_package'] == case_study['name']]
if len(br) > 0:
    br = br.iloc[0]
    print(f'\nBlast Radius (3 hops):')
    print(f'  Direct dependents (1-hop):    {int(br["direct_dependents"]):,}')
    print(f'  2-hop dependents:             {int(br["two_hop_dependents"]):,}')
    print(f'  3-hop dependents:             {int(br["three_hop_dependents"]):,}')
    print(f'  Total exposed packages:       {int(br["blast_radius_count"]):,}')

## 6 — Metric Correlation: PageRank vs Bus Factor

In [ ]:
# Scatter: PageRank (x) vs Bus Factor (y), colored by risk tier
plot_data = sentinel.dropna(subset=['pagerank', 'bus_factor']).copy()
# Cap bus_factor for readability
plot_data['bus_factor_capped'] = plot_data['bus_factor'].clip(upper=20)

fig, ax = plt.subplots(figsize=(11, 7))

for tier, color in RISK_COLORS.items():
    subset = plot_data[plot_data['risk_tier'] == tier]
    if len(subset) == 0:
        continue
    ax.scatter(
        subset['pagerank'],
        subset['bus_factor_capped'] + np.random.uniform(-0.2, 0.2, len(subset)),  # jitter
        c=color, label=tier, alpha=0.6, s=30, edgecolors='none'
    )

# Annotate the top 10 packages by sentinel score
for _, row in sentinel.nlargest(10, 'sentinel_score').iterrows():
    if pd.notna(row['pagerank']) and pd.notna(row['bus_factor']):
        ax.annotate(
            row['name'],
            (row['pagerank'], min(row['bus_factor'], 20)),
            fontsize=7, color='#333333',
            xytext=(5, 5), textcoords='offset points'
        )

ax.set_xlabel('PageRank (ecosystem centrality)', fontsize=11)
ax.set_ylabel('Bus Factor (active maintainers, capped at 20)', fontsize=11)
ax.set_title('PageRank vs Bus Factor\n(top-left = high centrality + low Bus Factor = highest risk)', fontsize=12)
ax.legend(title='Risk Tier', fontsize=9)

# Annotate danger zone
ax.axhline(1.5, color='red', linestyle='--', alpha=0.4, linewidth=1)
ax.text(ax.get_xlim()[1] * 0.01, 1.7, 'Bus Factor = 1  (danger zone)', color='red', fontsize=8)

plt.tight_layout()
plt.savefig(f'{FIG_DIR}/pagerank_vs_busfactor.png', bbox_inches='tight')
plt.show()
print(f'Figure saved → {FIG_DIR}/pagerank_vs_busfactor.png')

## 7 — Limitations & Future Work

**Current limitations:**

1. **Libraries.io data staleness (~2023):** The dependency graph reflects the state of the npm ecosystem as of the last Libraries.io snapshot. Package relationships change rapidly — new versions can remove or add dependencies. A production system would pull the npm registry directly via a streaming data pipeline.

2. **GitHub commit data ≠ npm publish rights:** A developer who commits to GitHub may not have npm publish access. Conversely, someone with npm publish access may never commit to the repository. Our Bus Factor (commit-based) and Orphan Risk (npm-maintainer-based) metrics each capture part of the picture. Future work: correlate the two to flag packages where the npm publisher is a different account from all GitHub committers (a red flag for account takeover).

3. **Version range matching for CVEs:** We flag packages if *any* version has a known CVE, regardless of the currently installed version. A production Sentinel would use the `semver` library to check whether the *latest stable version* is affected.

4. **Label Propagation vs Louvain:** Label Propagation can produce unstable communities (the result depends on the order nodes are processed). Louvain, which optimizes a global modularity objective, produces more stable and meaningful clusters. GraphFrames doesn't include Louvain natively, but it could be implemented with the `python-louvain` library after converting the graph to NetworkX.

5. **Scope: NPM only.** The methodology applies directly to PyPI (Python), Maven (Java), and RubyGems. The Log4Shell CVE that motivated this project was a Maven vulnerability. Extending to Maven would require re-running the pipeline with Libraries.io's Maven subset and the OSV Maven feed.

**High-value future work:**
- Real-time streaming pipeline using GitHub webhooks → Kafka → Spark Structured Streaming
- Integration with SBOM (Software Bill of Materials) standards (CycloneDX, SPDX) to score enterprise applications against their dependency trees
- Machine learning model to predict which Bus Factor=1 maintainers are likely to abandon a project (based on commit frequency trends, issue response rate)

In [ ]:
# Final summary printout (paste into your report writeup)
print('=' * 60)
print('FINAL REPORT SUMMARY')
print('=' * 60)
print(f'Total NPM packages analyzed:     {len(sentinel):,}')
print(f'Graph edges (DEPENDS_ON):        (see notebook 02)')
print(f'Packages with Bus Factor data:   {bus_factor["package_name"].nunique():,}')
print(f'Bus Factor = 1 (solo):           {(bus_factor["bus_factor"] == 1).sum():,}')
print(f'Packages with CVEs:              {sentinel["has_cve"].sum():,}')
print(f'CRITICAL packages:               {sentinel["is_critical"].sum():,}')
print(f'Orphan Risk flagged:             {orphan_risk["is_orphan_risk"].sum():,}')
print(f'Communities detected:            {communities["community_id"].nunique():,}')
print()
print('Top 5 highest Sentinel Score packages:')
for i, row in sentinel.nlargest(5, 'sentinel_score').iterrows():
    print(f'  #{i+1:<3} {row["name"]:<25} score={row["sentinel_score"]:.4f}  BF={int(row["bus_factor"])}  CVE={row["has_cve"]}')